# AI Resume Screening — End-to-End Demo

This notebook runs the full pipeline using the **Kaggle `snehaanbhawal/resume-dataset`**.
It demonstrates:
1. Dataset download via `kagglehub`
2. Text pre-processing (lower-casing, stopwords, lemmatisation)
3. Skill extraction with spaCy phrase matching
4. TF-IDF vectorisation
5. Cosine-similarity scoring
6. Candidate ranking
7. Skill-gap identification
8. Visualisations

In [ ]:
# ── Install / setup (run once) ───────────────────────────────────────────────
# !pip install -r ../requirements.txt
# !python -m spacy download en_core_web_sm
# import nltk; nltk.download('stopwords')

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
os.chdir('..')   # make relative paths in run_demo.py work

import kagglehub
from data_loader import load_kaggle_resumes, load_job_descriptions
from text_preprocessing import preprocess
from skill_extraction import extract_skills_from_dataframe
from vectorization import fit_vectorizer, transform_texts, save_vectorizer
from similarity_scoring import compute_similarity
from skill_gap_analysis import identify_missing_skills

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print('Imports OK')

In [ ]:
# ── 1. Load Data ─────────────────────────────────────────────────────────────
resumes = load_kaggle_resumes('snehaanbhawal/resume-dataset').head(100)
jobs    = load_job_descriptions('data/job_descriptions.csv')
print(resumes.shape, resumes.columns.tolist())
resumes.head(3)

In [ ]:
# ── 2. Pre-process ────────────────────────────────────────────────────────────
resumes['preprocessed_text'] = resumes['resume_text'].fillna('').apply(preprocess)
jobs['preprocessed_text']    = jobs['job_description'].fillna('').apply(preprocess)
resumes[['resume_id','preprocessed_text']].head(3)

In [ ]:
# ── 3. Skill Extraction ───────────────────────────────────────────────────────
skill_sets = extract_skills_from_dataframe(resumes, text_column='resume_text')
resumes['extracted_skills'] = [';'.join(sorted(s)) for s in skill_sets]
resumes[['resume_id','extracted_skills']].head(5)

In [ ]:
# ── 4. TF-IDF Vectorisation ───────────────────────────────────────────────────
corpus = pd.concat([resumes['preprocessed_text'], jobs['preprocessed_text']])
vectorizer = fit_vectorizer(corpus)
save_vectorizer(vectorizer, 'models/vectorizer.pkl')
resume_vectors = transform_texts(vectorizer, resumes['preprocessed_text'])
print('Vocabulary size:', len(vectorizer.vocabulary_))

In [ ]:
# ── 5 & 6. Similarity Scoring & Ranking ─────────────────────────────────────
job = jobs.iloc[0]
job_vec = transform_texts(vectorizer, pd.Series([job['preprocessed_text']]))
scores  = compute_similarity(resume_vectors, job_vec[0])

ranking = pd.DataFrame({'resume_id': resumes['resume_id'].values, 'score': scores})
ranking = ranking.sort_values('score', ascending=False).reset_index(drop=True)
ranking['rank'] = ranking.index + 1
print(f"Top 10 candidates for '{job['job_title']}':")
ranking.head(10)

In [ ]:
# ── 7. Skill Gap Analysis ────────────────────────────────────────────────────
req = set(s.strip().lower() for s in job['required_skills'].split(';'))
for _, row in ranking.head(5).iterrows():
    r   = resumes[resumes['resume_id'] == row['resume_id']].iloc[0]
    got = set(s.strip().lower() for s in r['extracted_skills'].split(';') if s)
    missing = identify_missing_skills(got, req)
    print(f"  Rank {row['rank']} | {row['resume_id']} | Missing: {missing or 'None'}")

In [ ]:
# ── 8. Visualisations ────────────────────────────────────────────────────────
os.makedirs('visualizations', exist_ok=True)

# Candidate scores
top20 = ranking.head(20)
plt.figure(figsize=(12, 5))
sns.barplot(data=top20, x='resume_id', y='score', palette='viridis')
plt.title(f"Top-20 Candidate Scores — {job['job_title']}")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('visualizations/candidate_scores.png', dpi=150)
plt.show()

# Skill distribution
all_skills = [s.lower() for ss in skill_sets for s in ss]
sc = pd.Series(all_skills).value_counts().head(15)
plt.figure(figsize=(10, 4))
sns.barplot(x=sc.index, y=sc.values, palette='magma')
plt.title('Top-15 Skill Frequency')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('visualizations/skill_distribution.png', dpi=150)
plt.show()

# Score distribution histogram
plt.figure(figsize=(8, 4))
sns.histplot(ranking['score'], bins=20, kde=True, color='steelblue')
plt.title('Resume Score Distribution')
plt.tight_layout()
plt.savefig('visualizations/ranking_chart.png', dpi=150)
plt.show()

In [ ]:
# ── 9. Save All Outputs ───────────────────────────────────────────────────────
import os, json, shutil
from datetime import datetime

OUT = "outputs"
os.makedirs(OUT, exist_ok=True)

# 1. Full ranking table
ranking.to_csv(f"{OUT}/ranking_table.csv", index=False)
print(f"Saved ranking_table.csv  ({len(ranking)} rows)")

# 2. Preprocessed resumes
resumes[["resume_id", "job_role", "extracted_skills", "preprocessed_text"]].to_csv(
    f"{OUT}/resumes_processed.csv", index=False)
print(f"Saved resumes_processed.csv  ({len(resumes)} rows)")

# 3. Skill gap report (top-20 candidates)
req = set(s.strip().lower() for s in job["required_skills"].split(";"))
gap_rows = []
for _, row in ranking.head(20).iterrows():
    r = resumes[resumes["resume_id"] == row["resume_id"]].iloc[0]
    got = set(s.strip().lower() for s in r["extracted_skills"].split(";") if s)
    missing = identify_missing_skills(got, req)
    gap_rows.append({
        "rank": row["rank"],
        "resume_id": row["resume_id"],
        "similarity_score": round(row["score"], 4),
        "matched_skills": "; ".join(sorted(got & req)) or "None",
        "missing_skills": "; ".join(sorted(missing)) or "None",
    })
pd.DataFrame(gap_rows).to_csv(f"{OUT}/skill_gap_report.csv", index=False)
print(f"Saved skill_gap_report.csv  ({len(gap_rows)} rows)")

# 4. Skill frequency table
all_skills = [s.lower() for ss in skill_sets for s in ss]
skill_freq = pd.Series(all_skills).value_counts().reset_index()
skill_freq.columns = ["skill", "frequency"]
skill_freq.to_csv(f"{OUT}/skill_frequency.csv", index=False)
print(f"Saved skill_frequency.csv  ({len(skill_freq)} unique skills)")

# 5. Summary stats JSON
stats = {
    "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "job_title": job["job_title"],
    "total_resumes": int(len(resumes)),
    "vocabulary_size": int(len(vectorizer.vocabulary_)),
    "top_candidate_id": str(ranking.iloc[0]["resume_id"]),
    "top_score": round(float(ranking.iloc[0]["score"]), 4),
    "mean_score": round(float(ranking["score"].mean()), 4),
    "median_score": round(float(ranking["score"].median()), 4),
    "std_score": round(float(ranking["score"].std()), 4),
    "unique_skills_found": int(len(skill_freq)),
    "required_skills": job["required_skills"],
}
with open(f"{OUT}/summary_stats.json", "w") as fh:
    json.dump(stats, fh, indent=2)

print("\n===== Summary Stats =====")
for k, v in stats.items():
    print(f"  {k:<25} {v}")

# 6. Copy visualisation PNGs into outputs/
for png in ["candidate_scores.png", "skill_distribution.png", "ranking_chart.png"]:
    src = f"visualizations/{png}"
    if os.path.exists(src):
        shutil.copy(src, f"{OUT}/{png}")
        print(f"Copied  {png}  →  {OUT}/")

print("\nAll outputs saved to:", os.path.abspath(OUT))
